# Lab 3 - hinet_lite_npu_v1

- Author: <your_name>
- Date: 2026-04-21
- Goal: improve PSNR while keeping NPU-friendly operator path
- Expected input/output: 256x256x3

This notebook is self-contained for Lab 3: setup, data audit, model/training, validation, ONNX export, calibration bundle, and MXQ handoff metadata.

## 1. Setup

Imports, Colab bootstrap installs, deterministic run config, Google Drive mount for Colab, configurable Drive data/run roots, optional checkpoint resume, and run artifact directory contract.

In [1]:
try:
    import google.colab  # type: ignore
    IN_COLAB_BOOTSTRAP = True
except Exception:
    IN_COLAB_BOOTSTRAP = False

if IN_COLAB_BOOTSTRAP:
    %pip install -q onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 114.9 MB/s eta 0:00:0000:0100:01


In [2]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import onnx
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


try:
    BICUBIC_RESAMPLE = Image.Resampling.BICUBIC
except AttributeError:
    BICUBIC_RESAMPLE = Image.BICUBIC

try:
    FLIP_LEFT_RIGHT = Image.Transpose.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.Transpose.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.Transpose.ROTATE_90
    ROTATE_180 = Image.Transpose.ROTATE_180
    ROTATE_270 = Image.Transpose.ROTATE_270
except AttributeError:
    FLIP_LEFT_RIGHT = Image.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.ROTATE_90
    ROTATE_180 = Image.ROTATE_180
    ROTATE_270 = Image.ROTATE_270


def now_stamp() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

In [3]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp"}
EXPECTED_TRAIN_PAIRS = 3036
EXPECTED_VAL_PAIRS = 100
DRIVE_ROOT = Path("/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3")
DRIVE_DATA_ROOT = DRIVE_ROOT / "Data"
DRIVE_RUNS_ROOT = DRIVE_ROOT / "runs"
DRIVE_HINET_LITE_ROOT = DRIVE_ROOT / "hinet_lite_npu_v1_full_20260421_213215"
DEFAULT_RESUME_CHECKPOINT_PATH = DRIVE_HINET_LITE_ROOT / "checkpoints" / "best.pth"

RESUME_CHECKPOINT_PATH: str | None = str(DEFAULT_RESUME_CHECKPOINT_PATH)
RESUME_STRICT = True


@dataclass
class RunConfig:
    model_id: str = "hinet_lite_npu_v1"
    run_mode: str = "full"
    seed: int = 1337
    train_patch_size: int = 224
    eval_size: int = 256
    channels: int = 32
    encoder_blocks: tuple[int, int] = (2, 2)
    bottleneck_blocks: int = 4
    decoder_blocks: tuple[int, int] = (2, 2)
    batch_size: int = 8
    epochs: int = 100
    lr: float = 2e-4
    weight_decay: float = 1e-4
    num_workers: int | None = None
    prefetch_factor: int = 2
    train_pair_limit: int | None = None
    val_pair_limit: int | None = None
    use_amp: bool = True
    use_ema: bool = True
    use_channels_last: bool = True
    use_tf32: bool = True
    ema_decay: float = 0.999


cfg = RunConfig()
assert cfg.eval_size == 256, "Lab 3 eval size must stay 256"
assert cfg.run_mode == "full", "This notebook is full-mode only"

def _resolve_repo_root(start: Path) -> Path:
    current = start.resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "Data").exists() and (candidate / "runs").exists():
            return candidate
    for candidate in candidates:
        if (candidate / "Data").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate repo root containing Data/. "
        f"Started search from: {start.resolve()}"
    )


def _is_colab_runtime() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401

        return True
    except Exception:
        return False


def _auto_num_workers(in_colab: bool) -> int:
    if sys.platform == "darwin":
        return 0
    cpu_count = os.cpu_count() or 2
    if in_colab:
        return min(4, max(cpu_count - 1, 2))
    return min(2, cpu_count)


in_colab = _is_colab_runtime()
if in_colab:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)

    data_root = DRIVE_DATA_ROOT
    runs_root = DRIVE_RUNS_ROOT
    if not data_root.exists():
        raise FileNotFoundError(f"Google Drive data root not found: {data_root}")
else:
    repo_root = _resolve_repo_root(Path.cwd())
    data_root = repo_root / "Data"
    runs_root = repo_root / "runs"
    if not data_root.exists():
        raise FileNotFoundError(f"Data root not found: {data_root}")

if cfg.num_workers is None:
    cfg.num_workers = _auto_num_workers(in_colab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    if cfg.use_tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")

started_day = datetime.now().strftime("%Y%m%d")
run_name = f"{cfg.model_id}_{cfg.run_mode}_{now_stamp()}"
run_root = runs_root / started_day / run_name
checkpoints_dir = run_root / "checkpoints"
exports_dir = run_root / "exports"
calibration_dir = exports_dir / "calibration"
summary_path = run_root / "summary.json"
metrics_csv_path = run_root / "metrics.csv"
epoch_logs_jsonl_path = run_root / "epoch_logs.jsonl"
history_json_path = run_root / "history.json"

for d in [run_root, checkpoints_dir, exports_dir, calibration_dir]:
    d.mkdir(parents=True, exist_ok=True)

set_global_seed(cfg.seed)
print(
    {
        "device": str(device),
        "run_mode": cfg.run_mode,
        "in_colab": in_colab,
        "data_root": str(data_root),
        "runs_root": str(runs_root),
        "run_root": str(run_root),
        "num_workers": cfg.num_workers,
        "prefetch_factor": cfg.prefetch_factor,
        "use_channels_last": cfg.use_channels_last,
        "resume_checkpoint_path": RESUME_CHECKPOINT_PATH,
    }
)

Mounted at /content/drive
{'device': 'cuda', 'run_mode': 'full', 'in_colab': True, 'data_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data', 'runs_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs', 'run_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544', 'num_workers': 4, 'prefetch_factor': 2, 'use_channels_last': True, 'resume_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/hinet_lite_npu_v1_full_20260421_213215/checkpoints/best.pth'}


### Colab Drive Data Staging (one-time)

Use this cell in Colab to validate your Google Drive Lab 3 `Data/` layout and optionally extract a dataset zip into the expected path.

Expected structure under `DRIVE_DATA_ROOT` (`.../Lab 3/Data`):
- `HR_train/HR_train1` ... `HR_train/HR_train4`
- `LR_train/LR_train1` ... `LR_train/LR_train4`
- `HR_val`
- `LR_val`

The default resume path points to the HiNet Lite checkpoint under `DRIVE_ROOT/hinet_lite_npu_v1_full_20260421_213215/checkpoints/best.pth`.

Datasets are still expected under `DRIVE_DATA_ROOT` (`.../Lab 3/Data`).

If you want a fresh run instead, set `RESUME_CHECKPOINT_PATH = None`.

In [4]:
# Optional one-time extraction helper if your dataset is uploaded as a zip.
# Set EXTRACT_DATASET_ZIP=True only when needed.
EXTRACT_DATASET_ZIP = False
DATASET_ZIP_PATH = DRIVE_DATA_ROOT / "dataset.zip"

required_dirs = [
    DRIVE_DATA_ROOT / "HR_train" / f"HR_train{i}" for i in range(1, 5)
] + [
    DRIVE_DATA_ROOT / "LR_train" / f"LR_train{i}" for i in range(1, 5)
] + [
    DRIVE_DATA_ROOT / "HR_val",
    DRIVE_DATA_ROOT / "LR_val",
]

if EXTRACT_DATASET_ZIP:
    import zipfile

    if not DATASET_ZIP_PATH.exists():
        raise FileNotFoundError(f"Dataset zip not found: {DATASET_ZIP_PATH}")
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zf:
        zf.extractall(DRIVE_DATA_ROOT)
    print(f"Extracted dataset archive into {DRIVE_DATA_ROOT}")

missing_dirs = [p for p in required_dirs if not p.exists()]
if missing_dirs:
    print("Missing required dataset directories:")
    for p in missing_dirs:
        print(f" - {p}")
else:
    print(f"Dataset layout looks good under: {DRIVE_DATA_ROOT}")

Dataset layout looks good under: /content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data


## 2. Data

Build synchronized train/val paired datasets from matching LR/HR basenames before creating loaders.

In [5]:
def scan_image_directory(directory: Path) -> dict[str, Path]:
    if not directory.exists():
        raise FileNotFoundError(f"Missing directory: {directory}")
    stem_to_path: dict[str, Path] = {}
    for path in directory.iterdir():
        if not path.is_file() or path.suffix.lower() not in IMAGE_SUFFIXES:
            continue
        stem = path.stem
        if stem not in stem_to_path:
            stem_to_path[stem] = path
    return stem_to_path


def collect_split_pairs(
    hr_dir: Path,
    lr_dir: Path,
    split_name: str,
) -> list[tuple[Path, Path, str]]:
    hr_map = scan_image_directory(hr_dir)
    lr_map = scan_image_directory(lr_dir)

    pairs: list[tuple[Path, Path, str]] = []
    for stem in sorted(set(hr_map) & set(lr_map)):
        if split_name.startswith("train"):
            pair_name = f"HR_{split_name}/{stem}"
        else:
            pair_name = stem
        pairs.append((lr_map[stem], hr_map[stem], pair_name))
    return pairs


def collect_all_pairs(data_root: Path) -> tuple[list[tuple[Path, Path, str]], list[tuple[Path, Path, str]]]:
    train_pairs: list[tuple[Path, Path, str]] = []
    for i in range(1, 5):
        train_pairs.extend(collect_split_pairs(
            data_root / "HR_train" / f"HR_train{i}",
            data_root / "LR_train" / f"LR_train{i}",
            split_name=f"train{i}",
        ))
    val_pairs = collect_split_pairs(
        data_root / "HR_val",
        data_root / "LR_val",
        split_name="val",
    )
    return train_pairs, val_pairs

In [6]:
class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], train: bool, patch_size: int, eval_size: int, seed: int):
        self.pairs = pairs
        self.train = train
        self.patch_size = patch_size
        self.eval_size = eval_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    @staticmethod
    def _to_tensor(img: Image.Image) -> torch.Tensor:
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

    def _augment_pair(self, lr: Image.Image, hr: Image.Image, index: int) -> tuple[Image.Image, Image.Image]:
        rng = random.Random((self.seed * 1_000_003) + (self.epoch * 9_973) + index)
        w, h = lr.size
        p = self.patch_size
        if w < p or h < p:
            lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            w, h = lr.size
        left = rng.randint(0, w - p)
        top = rng.randint(0, h - p)
        box = (left, top, left + p, top + p)
        lr = lr.crop(box)
        hr = hr.crop(box)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_LEFT_RIGHT)
            hr = hr.transpose(FLIP_LEFT_RIGHT)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_TOP_BOTTOM)
            hr = hr.transpose(FLIP_TOP_BOTTOM)
        rot = rng.choice([0, 1, 2, 3])
        if rot:
            rotate_ops = [ROTATE_90, ROTATE_180, ROTATE_270]
            lr = lr.transpose(rotate_ops[rot - 1])
            hr = hr.transpose(rotate_ops[rot - 1])
        return lr, hr

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Any]:
        lr_path, hr_path, name = self.pairs[index]
        lr = Image.open(lr_path).convert("RGB")
        hr = Image.open(hr_path).convert("RGB")
        if self.train:
            lr, hr = self._augment_pair(lr, hr, index)
        else:
            if lr.size != (self.eval_size, self.eval_size):
                lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
                hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
        return {"lr": self._to_tensor(lr), "hr": self._to_tensor(hr), "name": name}

In [7]:
train_pairs, val_pairs = collect_all_pairs(data_root)

if cfg.train_pair_limit is not None:
    train_pairs = train_pairs[: cfg.train_pair_limit]
if cfg.val_pair_limit is not None:
    val_pairs = val_pairs[: cfg.val_pair_limit]

if not train_pairs or not val_pairs:
    raise RuntimeError(
        f"Expected non-empty train/val pairs, found train={len(train_pairs)} val={len(val_pairs)}"
    )

train_ds = PairedSRDataset(train_pairs, train=True, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)
val_ds = PairedSRDataset(val_pairs, train=False, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)

pin_memory = device.type == "cuda"
loader_kwargs = {
    "num_workers": int(cfg.num_workers),
    "pin_memory": pin_memory,
}
if cfg.num_workers > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = cfg.prefetch_factor

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, **loader_kwargs)

sample = next(iter(val_loader))
assert tuple(sample["lr"].shape[1:]) == (3, 256, 256)
assert tuple(sample["hr"].shape[1:]) == (3, 256, 256)

print({
    "train_pairs": len(train_pairs),
    "val_pairs": len(val_pairs),
    "train_batches_per_epoch": len(train_loader),
    "val_batches_per_epoch": len(val_loader),
    "train_sample_shape": tuple(sample["lr"].shape),
    "device": str(device),
    "run_mode": cfg.run_mode,
    "num_workers": cfg.num_workers,
    "train_pair_limit": cfg.train_pair_limit,
    "val_pair_limit": cfg.val_pair_limit,
})

{'train_pairs': 3036, 'val_pairs': 100, 'train_batches_per_epoch': 380, 'val_batches_per_epoch': 13, 'train_sample_shape': (8, 3, 256, 256), 'device': 'cuda', 'run_mode': 'full', 'num_workers': 4, 'train_pair_limit': None, 'val_pair_limit': None}


## 3. Model and Training

Define HiNet-lite blocks and train with epoch-level logging for runtime and PSNR metrics.

In [8]:
@dataclass
class HiNetLiteConfig:
    channels: int = 32
    encoder_blocks: tuple[int, int] = (2, 2)
    bottleneck_blocks: int = 4
    decoder_blocks: tuple[int, int] = (2, 2)
    global_residual: bool = True


class HINResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        if channels % 2 != 0:
            raise ValueError(f"channels must be even, got {channels}")
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.inorm = nn.InstanceNorm2d(channels // 2, affine=True, track_running_stats=False)
        self.act1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.act2 = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        r = self.conv1(x)
        c = r.shape[1]
        r1, r2 = torch.split(r, [c // 2, c - c // 2], dim=1)
        r = torch.cat([self.inorm(r1), r2], dim=1)
        r = self.act1(r)
        r = self.act2(self.conv2(r))
        return x + r


class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


class DownsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class HiNetLiteSR(nn.Module):
    def __init__(self, cfg_m: HiNetLiteConfig):
        super().__init__()
        c = cfg_m.channels
        self.global_residual = cfg_m.global_residual

        self.stem = nn.Sequential(nn.Conv2d(3, c, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True))
        self.enc1 = nn.Sequential(*[HINResidualBlock(c) for _ in range(cfg_m.encoder_blocks[0])])
        self.down1 = DownsampleBlock(c, c * 2)
        self.enc2 = nn.Sequential(*[HINResidualBlock(c * 2) for _ in range(cfg_m.encoder_blocks[1])])
        self.down2 = DownsampleBlock(c * 2, c * 4)

        self.bottleneck = nn.Sequential(*[ResidualBlock(c * 4) for _ in range(cfg_m.bottleneck_blocks)])

        self.up1 = UpsampleBlock(c * 4, c * 2)
        self.dec1 = nn.Sequential(*[ResidualBlock(c * 2) for _ in range(cfg_m.decoder_blocks[0])])
        self.up2 = UpsampleBlock(c * 2, c)
        self.dec2 = nn.Sequential(*[ResidualBlock(c) for _ in range(cfg_m.decoder_blocks[1])])
        self.tail = nn.Conv2d(c, 3, 3, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = self.stem(x)
        e1 = self.enc1(s)
        d1 = self.down1(e1)
        e2 = self.enc2(d1)
        d2 = self.down2(e2)

        b = self.bottleneck(d2)
        u1 = self.up1(b)
        u1 = self.dec1(u1 + e2)
        u2 = self.up2(u1)
        u2 = self.dec2(u2 + e1)
        delta = self.tail(u2)
        return x + delta if self.global_residual else delta

In [9]:
def psnr_from_mse(mse: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return 10.0 * torch.log10(1.0 / torch.clamp(mse, min=eps))


@torch.no_grad()
def batch_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(pred, target, reduction="none").mean(dim=(1, 2, 3))
    return float(psnr_from_mse(mse).mean().item())


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow, strict=True)


def infer_resume_run_root(checkpoint_path: str | None) -> Path | None:
    if not checkpoint_path:
        return None
    ckpt_path = Path(checkpoint_path)
    if ckpt_path.parent.name != "checkpoints":
        return None
    return ckpt_path.parent.parent


def get_scaler_state(scaler: torch.cuda.amp.GradScaler) -> dict[str, Any] | None:
    return scaler.state_dict() if scaler.is_enabled() else None


def load_resume_checkpoint(
    checkpoint_path: str | None,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    ema: EMA | None,
    device: torch.device,
    strict: bool = True,
) -> tuple[int, float, int, dict[str, Any] | None]:
    if not checkpoint_path:
        return 0, -math.inf, 0, None

    ckpt_path = Path(checkpoint_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {ckpt_path}")

    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model"], strict=strict)

    if ema is not None and checkpoint.get("ema_model") is not None:
        ema.shadow = {
            k: v.detach().clone().to(device)
            for k, v in checkpoint["ema_model"].items()
        }

    if checkpoint.get("optimizer") is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])
    scaler_state = checkpoint.get("scaler")
    if scaler_state:
        if scaler.is_enabled():
            scaler.load_state_dict(scaler_state)
        else:
            print("Skipping checkpoint GradScaler state because the current scaler is disabled")
    elif scaler_state == {}:
        print("Skipping empty checkpoint GradScaler state")

    start_epoch = int(checkpoint.get("epoch", 0))
    best_val_psnr = float(checkpoint.get("best_val_psnr", -math.inf))
    best_epoch = int(checkpoint.get("best_epoch", 0))
    return start_epoch, best_val_psnr, best_epoch, checkpoint.get("last_metrics")


def load_history_json_rows(history_path: Path) -> list[dict[str, Any]]:
    payload = json.loads(history_path.read_text(encoding="utf-8"))
    rows = payload.get("epochs", [])
    return [{
        "epoch": int(row["epoch"]),
        "epoch_time_sec": float(row["epoch_time_sec"]),
        "train_loss": float(row["train_loss"]),
        "train_psnr": float(row["train_psnr"]),
        "val_psnr": float(row["val_psnr"]),
        "input_psnr": float(row["input_psnr"]),
        "delta_psnr": float(row["delta_psnr"]),
    } for row in rows]


def load_resume_history_rows(checkpoint_path: str | None) -> list[dict[str, Any]]:
    run_root = infer_resume_run_root(checkpoint_path)
    if run_root is None:
        return []

    history_path = run_root / "history.json"
    if history_path.exists():
        return load_history_json_rows(history_path)

    metrics_path = run_root / "metrics.csv"
    if not metrics_path.exists():
        return []

    with metrics_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = []
        for row in reader:
            rows.append({
                "epoch": int(row["epoch"]),
                "epoch_time_sec": float(row["epoch_time_sec"]),
                "train_loss": float(row["train_loss"]),
                "train_psnr": float(row["train_psnr"]),
                "val_psnr": float(row["val_psnr"]),
                "input_psnr": float(row["input_psnr"]),
                "delta_psnr": float(row["delta_psnr"]),
            })
    return rows


def write_history_json(history_path: Path, rows: list[dict[str, Any]], resume_checkpoint_path: str | None) -> None:
    payload = {
        "resume_checkpoint_path": resume_checkpoint_path,
        "epoch_count": len(rows),
        "latest_epoch": int(rows[-1]["epoch"]) if rows else 0,
        "best_val_psnr": max((float(row["val_psnr"]) for row in rows), default=None),
        "epochs": rows,
    }
    history_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def print_epoch_row(row: dict[str, Any]) -> None:
    print(
        f"[epoch {int(row['epoch']):03d}] "
        f"time={float(row['epoch_time_sec']):.2f}s "
        f"train_psnr={float(row['train_psnr']):.4f} "
        f"val_psnr={float(row['val_psnr']):.4f}"
    )


def move_image_batch(batch: torch.Tensor, device: torch.device) -> torch.Tensor:
    batch = batch.to(device, non_blocking=True)
    if cfg.use_channels_last and device.type == "cuda":
        batch = batch.contiguous(memory_format=torch.channels_last)
    return batch


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    device: torch.device,
    ema: EMA | None,
) -> tuple[float, float]:
    model.train()
    loss_meter = 0.0
    psnr_meter = 0.0
    count = 0

    for batch in loader:
        lr = move_image_batch(batch["lr"], device)
        hr = move_image_batch(batch["hr"], device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.use_amp and device.type == "cuda")):
            pred = model(lr)
            loss = F.l1_loss(pred, hr)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)

        with torch.no_grad():
            loss_meter += float(loss.item())
            psnr_meter += batch_psnr(pred.clamp(0, 1), hr)
            count += 1

    return loss_meter / max(count, 1), psnr_meter / max(count, 1)


@torch.inference_mode()
def validate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    val_psnr_meter = 0.0
    input_psnr_meter = 0.0
    count = 0

    for batch in loader:
        lr = move_image_batch(batch["lr"], device)
        hr = move_image_batch(batch["hr"], device)
        pred = model(lr).clamp(0, 1)
        val_psnr_meter += batch_psnr(pred, hr)
        input_psnr_meter += batch_psnr(lr, hr)
        count += 1

    val_psnr = val_psnr_meter / max(count, 1)
    input_psnr = input_psnr_meter / max(count, 1)
    return {
        "val_psnr": float(val_psnr),
        "input_psnr": float(input_psnr),
        "delta_psnr": float(val_psnr - input_psnr),
    }

In [10]:
model_cfg = HiNetLiteConfig(
    channels=cfg.channels,
    encoder_blocks=cfg.encoder_blocks,
    bottleneck_blocks=cfg.bottleneck_blocks,
    decoder_blocks=cfg.decoder_blocks,
)

model = HiNetLiteSR(model_cfg).to(device)
if cfg.use_channels_last and device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
shape_input = torch.randn(1, 3, 256, 256, device=device)
if cfg.use_channels_last and device.type == "cuda":
    shape_input = shape_input.contiguous(memory_format=torch.channels_last)
shape_check = model(shape_input).shape
assert tuple(shape_check) == (1, 3, 256, 256), f"Unexpected shape: {shape_check}"

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))
ema = EMA(model, decay=cfg.ema_decay) if cfg.use_ema else None
ema_eval_model = None
if ema is not None:
    ema_eval_model = HiNetLiteSR(model_cfg).to(device)
    if cfg.use_channels_last and device.type == "cuda":
        ema_eval_model = ema_eval_model.to(memory_format=torch.channels_last)

metrics_header = [
    "epoch",
    "epoch_time_sec",
    "train_loss",
    "train_psnr",
    "val_psnr",
    "input_psnr",
    "delta_psnr",
]

best_ckpt_path = checkpoints_dir / "best.pth"
last_ckpt_path = checkpoints_dir / "last.pth"

start_epoch, best_val_psnr, best_epoch, last_metrics = load_resume_checkpoint(
    RESUME_CHECKPOINT_PATH,
    model,
    optimizer,
    scaler,
    ema,
    device,
    strict=RESUME_STRICT,
)
resume_history_rows = load_resume_history_rows(RESUME_CHECKPOINT_PATH)
history_rows = [dict(row) for row in resume_history_rows]

with metrics_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=metrics_header)
    writer.writeheader()
    for row in resume_history_rows:
        writer.writerow(row)

with epoch_logs_jsonl_path.open("w", encoding="utf-8") as f:
    for row in resume_history_rows:
        f.write(json.dumps(row) + "\n")

write_history_json(history_json_path, history_rows, RESUME_CHECKPOINT_PATH)

if resume_history_rows:
    print("Replaying prior run history from history.json/metrics.csv")
    for row in resume_history_rows:
        print_epoch_row(row)
elif last_metrics is not None:
    print("No metrics.csv found for resume run; using checkpoint last_metrics")
    print_epoch_row(last_metrics)

for epoch in range(start_epoch, cfg.epochs):
    train_ds.set_epoch(epoch)
    t0 = time.perf_counter()
    train_loss, train_psnr = train_one_epoch(model, train_loader, optimizer, scaler, device, ema)

    eval_model = model
    if ema is not None and ema_eval_model is not None:
        ema.copy_to(ema_eval_model)
        eval_model = ema_eval_model

    val_metrics = validate(eval_model, val_loader, device)
    epoch_time_sec = float(time.perf_counter() - t0)

    row = {
        "epoch": epoch + 1,
        "epoch_time_sec": epoch_time_sec,
        "train_loss": float(train_loss),
        "train_psnr": float(train_psnr),
        "val_psnr": float(val_metrics["val_psnr"]),
        "input_psnr": float(val_metrics["input_psnr"]),
        "delta_psnr": float(val_metrics["delta_psnr"]),
    }

    with metrics_csv_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=metrics_header)
        writer.writerow(row)

    with epoch_logs_jsonl_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")

    history_rows.append(row)
    write_history_json(history_json_path, history_rows, RESUME_CHECKPOINT_PATH)

    # Always persist the latest epoch state.
    torch.save({
        "epoch": row["epoch"],
        "model": model.state_dict(),
        "ema_model": (ema.shadow if ema is not None else None),
        "optimizer": optimizer.state_dict(),
        "scaler": get_scaler_state(scaler),
        "config": asdict(cfg),
        "model_config": asdict(model_cfg),
        "best_val_psnr": float(best_val_psnr),
        "best_epoch": int(best_epoch),
        "last_metrics": row,
    }, last_ckpt_path)

    if row["val_psnr"] > best_val_psnr:
        best_val_psnr = row["val_psnr"]
        best_epoch = row["epoch"]
        torch.save({
            "epoch": row["epoch"],
            "model": model.state_dict(),
            "ema_model": (ema.shadow if ema is not None else None),
            "optimizer": optimizer.state_dict(),
            "scaler": get_scaler_state(scaler),
            "config": asdict(cfg),
            "model_config": asdict(model_cfg),
            "best_val_psnr": float(best_val_psnr),
            "best_epoch": int(best_epoch),
            "last_metrics": row,
        }, best_ckpt_path)

    print_epoch_row(row)

print({
    "best_epoch": best_epoch,
    "best_val_psnr": best_val_psnr,
    "best_ckpt_path": str(best_ckpt_path),
    "last_ckpt_path": str(last_ckpt_path),
})

/tmp/ipykernel_7867/275085355.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


Skipping empty checkpoint GradScaler state
Replaying prior run history from history.json/metrics.csv
[epoch 001] time=645.22s train_psnr=25.5856 val_psnr=15.4651
[epoch 002] time=691.46s train_psnr=26.5955 val_psnr=18.5029
[epoch 003] time=659.78s train_psnr=26.7227 val_psnr=20.1901
[epoch 004] time=640.92s train_psnr=26.8204 val_psnr=20.9367
[epoch 005] time=647.84s train_psnr=26.9324 val_psnr=21.2430
[epoch 006] time=640.49s train_psnr=27.0021 val_psnr=21.3864
[epoch 007] time=688.17s train_psnr=27.0705 val_psnr=21.4718
[epoch 008] time=838.78s train_psnr=27.1228 val_psnr=21.5398
[epoch 009] time=791.45s train_psnr=27.1568 val_psnr=21.6019
[epoch 010] time=829.52s train_psnr=27.2255 val_psnr=21.6602
[epoch 011] time=817.88s train_psnr=27.2636 val_psnr=21.7133
[epoch 012] time=903.32s train_psnr=27.3026 val_psnr=21.7603
[epoch 013] time=866.18s train_psnr=27.3150 val_psnr=21.8013
[epoch 014] time=753.11s train_psnr=27.3697 val_psnr=21.8380
[epoch 015] time=743.04s train_psnr=27.3711 v

/tmp/ipykernel_7867/440607702.py:169: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(cfg.use_amp and device.type == "cuda")):


[epoch 067] time=1334.31s train_psnr=27.8275 val_psnr=22.3077
[epoch 068] time=16.41s train_psnr=27.8607 val_psnr=22.3106
[epoch 069] time=17.51s train_psnr=27.8755 val_psnr=22.3115
[epoch 070] time=17.36s train_psnr=27.8982 val_psnr=22.3121
[epoch 071] time=17.55s train_psnr=27.9096 val_psnr=22.3115
[epoch 072] time=16.84s train_psnr=27.9371 val_psnr=22.3102
[epoch 073] time=16.62s train_psnr=27.9564 val_psnr=22.3081
[epoch 074] time=16.59s train_psnr=27.9679 val_psnr=22.3066
[epoch 075] time=16.77s train_psnr=27.9897 val_psnr=22.3034
[epoch 076] time=16.75s train_psnr=28.0020 val_psnr=22.2992
[epoch 077] time=16.81s train_psnr=28.0088 val_psnr=22.2960
[epoch 078] time=16.84s train_psnr=28.0320 val_psnr=22.2918
[epoch 079] time=16.85s train_psnr=28.0564 val_psnr=22.2865
[epoch 080] time=16.84s train_psnr=28.0670 val_psnr=22.2822
[epoch 081] time=16.75s train_psnr=28.0803 val_psnr=22.2774
[epoch 082] time=16.65s train_psnr=28.0985 val_psnr=22.2716
[epoch 083] time=16.78s train_psnr=28.

## 4. Validation

Report `val_psnr`, `input_psnr`, and `delta_psnr`, and save a validation preview triplet with sample identity.

In [11]:
def tensor_to_uint8_image(x: torch.Tensor) -> Image.Image:
    x = x.detach().cpu().clamp(0, 1)
    arr = (x.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)


@torch.no_grad()
def write_val_preview(model: nn.Module, loader: DataLoader, out_dir: Path, device: torch.device) -> dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    batch = next(iter(loader))
    lr = batch["lr"].to(device)
    hr = batch["hr"].to(device)
    name = batch["name"][0]
    pred = model(lr).clamp(0, 1)

    lr_path = out_dir / "val_preview_input.png"
    pred_path = out_dir / "val_preview_pred.png"
    hr_path = out_dir / "val_preview_target.png"

    tensor_to_uint8_image(lr[0]).save(lr_path)
    tensor_to_uint8_image(pred[0]).save(pred_path)
    tensor_to_uint8_image(hr[0]).save(hr_path)
    return {
        "sample_name": str(name),
        "input": str(lr_path),
        "prediction": str(pred_path),
        "target": str(hr_path),
    }


preview_info = write_val_preview(model, val_loader, exports_dir / "val_preview", device)
print(preview_info)

{'sample_name': '0000', 'input': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/exports/val_preview/val_preview_input.png', 'prediction': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/exports/val_preview/val_preview_pred.png', 'target': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/exports/val_preview/val_preview_target.png'}


## 5. Export and Submission Artifacts

### 5.1 ONNX Export and Operator Audit

In [12]:
onnx_path = exports_dir / f"{cfg.model_id}.onnx"
mxq_target_path = exports_dir / f"{cfg.model_id}.mxq"

checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint["model"], strict=True)
model.eval()

dummy = torch.randn(1, 3, 256, 256, device=device)
torch.onnx.export(
    model,
    dummy,
    str(onnx_path),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamo=False,  # use legacy exporter for local stability
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

node_ops = [node.op_type for node in onnx_model.graph.node]
node_ops_set = set(node_ops)
op_audit_map = {
    "Softmax": "Softmax",
    "MatMul": "MatMul",
    "Multiply": "Mul",  # ONNX uses Mul for element-wise multiply
}
blocked_ops = [label for label, onnx_name in op_audit_map.items() if onnx_name in node_ops_set]
if blocked_ops:
    raise RuntimeError(f"Unexpected attention-style ops in ONNX graph: {blocked_ops}")

op_audit = {
    "checked_ops": list(op_audit_map.keys()),
    "blocked_ops_found": blocked_ops,
    "graph_op_count": len(node_ops),
}
print({"onnx_path": str(onnx_path), **op_audit})

/tmp/ipykernel_7867/4087324342.py:9: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2809: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


{'onnx_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/exports/hinet_lite_npu_v1.onnx', 'checked_ops': ['Softmax', 'MatMul', 'Multiply'], 'blocked_ops_found': [], 'graph_op_count': 106}


In [13]:
# 5.2 CPU parity check (PyTorch vs ONNX Runtime)
parity_result = {
    "attempted": False,
    "onnxruntime_available": False,
    "passed": False,
    "max_abs_diff": None,
}

try:
    import onnxruntime as ort

    parity_result["attempted"] = True
    parity_result["onnxruntime_available"] = True

    cpu_model = HiNetLiteSR(model_cfg).cpu().eval()
    cpu_model.load_state_dict(checkpoint["model"], strict=True)
    parity_input = torch.randn(1, 3, 256, 256)
    with torch.no_grad():
        torch_out = cpu_model(parity_input).detach().cpu().numpy()

    ort_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    ort_out = ort_session.run(["output"], {"input": parity_input.numpy()})[0]
    max_abs_diff = float(np.max(np.abs(torch_out - ort_out)))

    parity_result["max_abs_diff"] = max_abs_diff
    parity_result["passed"] = bool(max_abs_diff < 1e-3)
    if not parity_result["passed"]:
        raise RuntimeError(f"ONNX parity failed: max_abs_diff={max_abs_diff:.6f}")
except ImportError:
    print("onnxruntime not installed; skipping CPU parity check")

print(parity_result)

onnxruntime not installed; skipping CPU parity check
{'attempted': False, 'onnxruntime_available': False, 'passed': False, 'max_abs_diff': None}


### 5.3 Calibration and Final Handoff

Build train-derived calibration samples and write final run summary with metrics, export checks, and artifact paths.

In [14]:
def write_calibration_bundle(train_pairs: list[tuple[Path, Path, str]], out_dir: Path, max_items: int = 32) -> dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    selected = train_pairs[: max_items]
    manifest = {
        "source": "train_pairs_only",
        "count": len(selected),
        "items": [],
    }
    for idx, (lr_path, _, name) in enumerate(selected):
        dst = out_dir / f"cal_{idx:04d}.png"
        Image.open(lr_path).convert("RGB").save(dst)
        manifest["items"].append({"name": name, "lr_source": str(lr_path), "cal_path": str(dst)})
    manifest_path = out_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return {"calibration_dir": str(out_dir), "manifest_path": str(manifest_path), "count": len(selected)}


cal_info = write_calibration_bundle(train_pairs, calibration_dir, max_items=32)
summary = {
    "model_id": cfg.model_id,
    "run_mode": cfg.run_mode,
    "config": asdict(cfg),
    "resume": {
        "resumed_from_checkpoint": RESUME_CHECKPOINT_PATH is not None,
        "resume_checkpoint_path": RESUME_CHECKPOINT_PATH,
        "resume_history_replayed": bool(resume_history_rows),
        "start_epoch": start_epoch,
    },
    "pair_summary": {
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "train_preview": [name for _, _, name in train_pairs[:3]],
        "val_preview": [name for _, _, name in val_pairs[:3]],
    },
    "latest_epoch": cfg.epochs,
    "best_val_psnr": float(best_val_psnr),
    "best_epoch": int(best_epoch),
    "metrics_csv_path": str(metrics_csv_path),
    "epoch_logs_jsonl_path": str(epoch_logs_jsonl_path),
    "history_json_path": str(history_json_path),
    "best_checkpoint_path": str(best_ckpt_path),
    "last_checkpoint_path": str(last_ckpt_path),
    "onnx_path": str(onnx_path),
    "op_audit": op_audit,
    "parity_result": parity_result,
    "calibration": cal_info,
    "mxq_target_path": str(mxq_target_path),
    "val_preview": preview_info,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print({
    "summary_path": str(summary_path),
    "best_checkpoint": str(best_ckpt_path),
    "onnx": str(onnx_path),
    "mxq_target": str(mxq_target_path),
})

{'summary_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/summary.json', 'best_checkpoint': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/checkpoints/best.pth', 'onnx': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/exports/hinet_lite_npu_v1.onnx', 'mxq_target': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_full_20260423_033544/exports/hinet_lite_npu_v1.mxq'}
